In [ ]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

In [ ]:
import os 

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

from src.utils.sampling_utils import TimeSeriesDataset
from src.experiments.ppg.ppg_model import PPGModel
from src.experiments.ppg.ppg_data_utils import * 

In [ ]:
# SETTINGS

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# data quality
INCLUDE_CLEAN_DATA = True
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False
TRAIN_SET_SIZE = int(1e4)

# logging 
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg"

# architecture 
REPR_DIMS = [10, 50, 100]
MODELS = {}
for dim in REPR_DIMS:
    model_list = {}
    
    ckpt_dir = f"{SAVE_DIR}/checkpoints/dim{dim}"
    for ckpt_file in os.listdir(ckpt_dir):
        epoch = ckpt_file[10:-3]
        ckpt = torch.load(f"{ckpt_dir}/{ckpt_file}", map_location = "cpu")
        MODEL = PPGModel(representation_dimension = dim)
        MODEL.load_state_dict(ckpt)
        MODEL = MODEL.to(DEVICE)
        if torch.cuda.device_count() > 1:
            MODEL = nn.DataParallel(MODEL)
        elif not isinstance(MODEL, nn.DataParallel):
            MODEL = nn.DataParallel(MODEL)
        MODEL.eval()

        model_list[epoch] = MODEL
    
    MODELS[dim] = model_list 

In [ ]:
# LOAD DATA

X_train, y_train = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = True
)
indices = np.random.default_rng(SEED).permutation(X_train.shape[0])
X_train, y_train = X_train[indices][:TRAIN_SET_SIZE], y_train[indices][:TRAIN_SET_SIZE]

X_test, y_test = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'test',
    return_labels = True
)

print(f'X_train shape: {X_train.shape}')
print(f'Train set positive samples: {np.sum(y_train)}')
print(f'X_test shape: {X_test.shape}')
print(f'Test set positive samples: {np.sum(y_test)}')

# RESHAPE HERE WHEN collate_fn = None 
X_train, X_test = np.transpose(X_train, (0, 2, 1)), np.transpose(X_test, (0, 2, 1))

In [ ]:
# GENERATE REPRESENTATIONS

with torch.inference_mode():
    
    X_train_tensor, X_test_tensor = torch.tensor(X_train, dtype = torch.float32), torch.tensor(X_test, dtype = torch.float32)

    for dim in REPR_DIMS:
        model = MODELS[dim]['epoch15']
        repr_train, repr_test = model(X_train_tensor), model(X_test_tensor)
        repr_train, repr_test = repr_train.cpu().numpy(), repr_test.cpu().numpy()

        np.savetxt(f'SAVE_DIR/test_reprs/prototsrl{dim}-repr-train.csv', repr_train, delimiter = ',')
        np.savetxt(f'SAVE_DIR/test_reprs/prototsrl{dim}-repr-test.csv', repr_test, delimiter = ',')

In [ ]:
# LOAD TRAIN AND TEST SET REPRESENTATIONS AND CREATE DATALOADERS

repr = {}
for dim in REPR_DIMS: 
    repr[dim] = {}
    for repr_set in ['train', 'test']:
        repr[dim][repr_set] = pd.read_csv(f'SAVE_DIR/test_reprs/prototsrl{dim}-repr-{repr_set}.csv', header = None)

# Convert to tensors 
y_tensors = {}
y_tensors['train'] = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
y_tensors['test'] = torch.tensor(y_test, dtype = torch.float32).unsqueeze(1)

repr_loaders = {}
for dim in REPR_DIMS: 
    repr_loaders[dim] = {}
    for repr_set in ['train', 'test']:
        X_tensor = torch.tensor(repr[dim][repr_set], dtype = torch.float32)
        X_dataset = TensorDataset(X_tensor, y_tensors[repr_set])
        if repr_set == 'train':
            repr_loaders[dim][repr_set] = DataLoader(X_dataset, batch_size = 256, shuffle = True)
        else: 
            repr_loaders[dim][repr_set] = DataLoader(X_dataset, batch_size = X_tensor.size(0), shuffle = False)

In [ ]:
# SIMPLE MODELS 

lr_clf = LogisticRegression(max_iter = 500, random_state = SEED)
dt_clf = DecisionTreeClassifier(min_samples_split = 5)
rf_clf = RandomForestClassifier(n_estimators = 100)

sample_weights = compute_sample_weight(class_weight = "balanced", y = y_train)

for dim in REPR_DIMS: 
    print(f'===== {dim} dimensional representations =====')
    
    df_train, df_test = repr[dim]['train'], repr[dim]['test']
    ss = StandardScaler().fit(df_train)
    df_train, df_test = ss.transform(df_train), ss.transform(df_test)
    
    lr_clf.fit(df_train, y_train, sample_weight = sample_weights)
    dt_clf.fit(df_train, y_train, sample_weight = sample_weights)
    rf_clf.fit(df_train, y_train, sample_weight = sample_weights)

    y_pred_lr = lr_clf.predict(df_test)
    y_pred_dt = dt_clf.predict(df_test)
    y_pred_rf = rf_clf.predict(df_test)

    acc_lr = accuracy_score(y_test, y_pred_lr)
    acc_dt = accuracy_score(y_test, y_pred_dt)
    acc_rf = accuracy_score(y_test, y_pred_rf)

    print(f'acc_lr = {round(acc_lr, 3)} | acc_dt = {round(acc_dt, 3)} | acc_rf = {round(acc_rf, 3)}')
    print(confusion_matrix(y_test, y_pred_lr), confusion_matrix(y_test, y_pred_dt, y_pred_rf))

In [ ]:
# MLP

class mlp_clf(nn.Module):
    def __init__(self, input_dim, hidden_dim1 = 128, hidden_dim2 = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, 1)
        )

    def forward(self, x):
        return self.net(x)  # logits

mlps = {}
opts = {}
for dim in REPR_DIMS: 
    mlps[dim] = mlp_clf(input_dim = dim)
    opts[dim] = torch.optim.Adam(mlps[dim].parameters(), lr = 1e-3)

# Class imbalance handling: pos_weight = (# negatives / # positives)
classes = np.unique(y_train.astype(int))
class_weights = compute_class_weight(
    class_weight = "balanced",
    classes = classes,
    y = y_train.astype(int)
)

# compute pos_weight manually for BCEWithLogitsLoss
n_neg, n_pos = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_neg / n_pos], dtype = torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)

def mlpclf_train(
        model,
        dataloader,
        device,
        criterion,
        optimizer,
        n_epochs
    ):

    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0

        for xb, yb in dataloader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1:02d}/{n_epochs} - Train Loss: {epoch_loss:.4f}")

def mlpclf_eval(
        model,
        dataloader,
        device
    ):

    model.eval()

    all_probs = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).float()

            all_probs.extend(probs.cpu().numpy().ravel())
            all_preds.extend(preds.cpu().numpy().ravel())
            all_targets.extend(yb.numpy().ravel())

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    print("\n=== MLP Binary Classifier ===")
    print(f"Accuracy          : {accuracy_score(all_targets, all_preds):.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_targets, all_preds))